In [9]:
from the_well.data.datasets import WellDataset
from pathlib import Path
import sys
import os

PROJECT_ROOT = "/glade/work/gaccarino/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.mini_well import create_mini_well

In [10]:
# Cell 2 — Inspect original dataset
#well_base_path    = '/Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets'
well_base_path    = '/glade/work/gaccarino/data/datasets'
#well_dataset_name = 'turbulent_radiative_layer_2D'
well_dataset_name = 'rayleigh_benard'
#well_dataset_name = 'planetswe'

MINI_DATASET  = f'{well_dataset_name}_mini'

for split in ['train', 'valid', 'test']:
    ds = WellDataset(
        well_base_path=well_base_path,
        well_dataset_name=well_dataset_name,
        well_split_name=split,
    )
    print(f"\n── {split} ──")
    print(f"  Spatial resolution   : {ds.size_tuple}")
    print(f"  N files              : {ds.n_files}")
    print(f"  Trajectories/file    : {ds.n_trajectories_per_file}")
    print(f"  Steps/traj           : {ds.n_steps_per_trajectory}")
    print(f"  Total trajectories   : {sum(ds.n_trajectories_per_file)}")


── train ──
  Spatial resolution   : (512, 128)
  N files              : 35
  Trajectories/file    : [40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40]
  Steps/traj           : [200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200]
  Total trajectories   : 1400

── valid ──
  Spatial resolution   : (512, 128)
  N files              : 35
  Trajectories/file    : [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
  Steps/traj           : [200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 200]
  Total trajectories   : 175

── test ──
  Spatial resolution   : (512, 128)
  N files              : 35
  Trajectori

In [11]:
# ── Parameter subset ─────────────────────────────────────────
ds_full = WellDataset(
    well_base_path=well_base_path,
    well_dataset_name=well_dataset_name,
    well_split_name='train',  # names are identical across splits
)
all_files = sorted(ds_full.files_paths)

KEEP_RA = ['1e6', '1e8', '1e10']   # soft / hard / ultimate turbulence
KEEP_PR = ['1e-1', '1', '10']      # low / mid / high Prandtl

# Anchor Pr match to .hdf5 suffix to avoid '1' matching '10' or '1e-1'
selected_names = [
    Path(f).name for f in all_files
    if any(f'Rayleigh_{ra}_' in Path(f).name for ra in KEEP_RA)
    and any(f'Prandtl_{pr}.hdf5' in Path(f).name for pr in KEEP_PR)
]
print(f"Keeping {len(selected_names)} files:")
for name in selected_names:
    print(f"  {name}")

Keeping 9 files:
  rayleigh_benard_Rayleigh_1e10_Prandtl_1.hdf5
  rayleigh_benard_Rayleigh_1e10_Prandtl_10.hdf5
  rayleigh_benard_Rayleigh_1e10_Prandtl_1e-1.hdf5
  rayleigh_benard_Rayleigh_1e6_Prandtl_1.hdf5
  rayleigh_benard_Rayleigh_1e6_Prandtl_10.hdf5
  rayleigh_benard_Rayleigh_1e6_Prandtl_1e-1.hdf5
  rayleigh_benard_Rayleigh_1e8_Prandtl_1.hdf5
  rayleigh_benard_Rayleigh_1e8_Prandtl_10.hdf5
  rayleigh_benard_Rayleigh_1e8_Prandtl_1e-1.hdf5


In [12]:
# ── Resampling config ─────────────────────────────────────────
SPATIAL_FACTOR = 1    # no spatial downsampling
TIME_FRACTION  = 1.0  # use all steps per trajectory
TIME_FACTOR    = 1    # no temporal downsampling

MAX_TRAJ_PER_FILE = {'train': 10, 'valid': 5, 'test': 5}

In [13]:
# ── Build mini dataset ────────────────────────────────────────
for split, max_traj in MAX_TRAJ_PER_FILE.items():
    print(f"\n{'='*50}  {split}  {'='*50}")
    for fname in selected_names:
        ds_single = WellDataset(
            well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name=split,
            include_filters=[fname],
        )
        create_mini_well(
            dataset=ds_single,
            well_base_path=well_base_path,
            mini_dataset_name=MINI_DATASET,
            spatial_downsample_factor=SPATIAL_FACTOR,
            time_downsample_factor=TIME_FACTOR,
            max_trajectories=max_traj,
            split=split,
            time_fraction=TIME_FRACTION,
        )

print("\nAll splits done ✓")


==================================================  train  ==================================================
  Copied stats.yaml → /glade/work/gaccarino/data/datasets/rayleigh_benard_mini/stats.yaml


train: 100%|██████████| 1/1 [00:06<00:00,  6.02s/it]

    rayleigh_benard_Rayleigh_1e10_Prandtl_1.hdf5: 10 traj written
  ✓ train: 10 trajectories → /glade/work/gaccarino/data/datasets/rayleigh_benard_mini/data/train



train: 100%|██████████| 1/1 [00:02<00:00,  2.35s/it]


    rayleigh_benard_Rayleigh_1e10_Prandtl_10.hdf5: 10 traj written
  ✓ train: 10 trajectories → /glade/work/gaccarino/data/datasets/rayleigh_benard_mini/data/train


train: 100%|██████████| 1/1 [00:02<00:00,  2.72s/it]


... (45 outputs left out)

test: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]

    rayleigh_benard_Rayleigh_1e8_Prandtl_10.hdf5: 5 traj written
  ✓ test: 5 trajectories → /glade/work/gaccarino/data/datasets/rayleigh_benard_mini/data/test



test: 100%|██████████| 1/1 [00:00<00:00,  1.31it/s]

    rayleigh_benard_Rayleigh_1e8_Prandtl_1e-1.hdf5: 5 traj written
  ✓ test: 5 trajectories → /glade/work/gaccarino/data/datasets/rayleigh_benard_mini/data/test

All splits done ✓


In [14]:
# Cell 5 — Verify the mini dataset
for split in ['train', 'valid', 'test']:
    ds_mini = WellDataset(
        path=os.path.join(well_base_path, MINI_DATASET, 'data', split), 
        #well_base_path=well_base_path,
        #well_dataset_name=MINI_DATASET,
        #well_split_name=split,
    )
    total_traj = sum(ds_mini.n_trajectories_per_file)
    print(f"\n── mini/{split} ──")
    print(f"  Spatial resolution : {ds_mini.size_tuple}")
    print(f"  N files            : {ds_mini.n_files}")
    print(f"  Trajectories/file  : {ds_mini.n_trajectories_per_file}")
    print(f"  Steps/traj         : {ds_mini.n_steps_per_trajectory}")
    print(f"  Total trajectories : {total_traj}")
    print(f"  Total windows      : {len(ds_mini)}")


── mini/train ──
  Spatial resolution : (512, 128)
  N files            : 9
  Trajectories/file  : [10, 10, 10, 10, 10, 10, 10, 10, 10]
  Steps/traj         : [200, 200, 200, 200, 200, 200, 200, 200, 200]
  Total trajectories : 90
  Total windows      : 17910

── mini/valid ──
  Spatial resolution : (512, 128)
  N files            : 9
  Trajectories/file  : [5, 5, 5, 5, 5, 5, 5, 5, 5]
  Steps/traj         : [200, 200, 200, 200, 200, 200, 200, 200, 200]
  Total trajectories : 45
  Total windows      : 8955

── mini/test ──
  Spatial resolution : (512, 128)
  N files            : 9
  Trajectories/file  : [5, 5, 5, 5, 5, 5, 5, 5, 5]
  Steps/traj         : [200, 200, 200, 200, 200, 200, 200, 200, 200]
  Total trajectories : 45
  Total windows      : 8955


In [15]:
"""ds_train = WellDataset(path=os.path.join(well_base_path, 'rayleigh_benard', 'data', 'train'))
print("Train files:")
for f in sorted(ds_train.files_paths): print(Path(f).name)

ds_valid = WellDataset(path=os.path.join(well_base_path, 'rayleigh_benard', 'data', 'valid'))
print("Valid files:")
for f in sorted(ds_valid.files_paths): print(Path(f).name)"""

'ds_train = WellDataset(path=os.path.join(well_base_path, \'rayleigh_benard\', \'data\', \'train\'))\nprint("Train files:")\nfor f in sorted(ds_train.files_paths): print(Path(f).name)\n\nds_valid = WellDataset(path=os.path.join(well_base_path, \'rayleigh_benard\', \'data\', \'valid\'))\nprint("Valid files:")\nfor f in sorted(ds_valid.files_paths): print(Path(f).name)'